# PyTorch, task by task

Run the two cells below (Shift+Enter). Then click **Next →** to go through the tasks: what we want to do, what that needs, what PyTorch gives us, the code, and the code running. You can edit the code in its box and press **Run ▶**. The slides *PyTorch, by Training a Language Model* explain every task in full.

In [ ]:
# @title Setup: run this cell first (the data, the pictures and the stepper)
# Setup: the names, and the helpers that draw the pictures. Run this cell first.
import os, urllib.request
if not os.path.exists("names.txt"):
    urllib.request.urlretrieve("https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/names.txt", "names.txt")

import math
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(1)

def show_emb(E, mark=(), grad=None):
    """The table E: one point per character (vowels red, consonants blue, '.' green)."""
    E = E.detach(); mark = set(int(i) for i in mark)
    fig, ax = plt.subplots(figsize=(5, 4.2))
    for i, c in enumerate(chars):
        col = "green" if c == "." else ("crimson" if c in "aeiou" else "royalblue")
        ax.text(E[i, 0], E[i, 1], c, color=col, ha="center", va="center", fontsize=13,
                fontweight="bold" if i in mark else "normal",
                bbox=dict(boxstyle="circle", fc="#fff1c2", ec="orange") if i in mark else None)
    if grad is not None:   # each row moves against its gradient
        G = -grad.detach()
        ax.quiver(E[:, 0], E[:, 1], G[:, 0], G[:, 1], color="gray", angles="xy", width=0.004)
    pad = 0.3
    ax.set_xlim(E[:, 0].min() - pad, E[:, 0].max() + pad); ax.set_ylim(E[:, 1].min() - pad, E[:, 1].max() + pad)
    ax.axhline(0, color="#eee", zorder=0); ax.axvline(0, color="#eee", zorder=0)
    ax.set_title("the table E: one point per character", fontsize=11)
    plt.show()

def show_bars(p, ctx, y=None):
    """P(next character | context) for the 27 characters; the true next one in red."""
    p = p.detach()
    fig, ax = plt.subplots(figsize=(8, 2.6))
    cols = ["crimson" if y is not None and i == int(y) else "royalblue" for i in range(len(chars))]
    ax.bar(range(len(chars)), p, color=cols)
    ax.set_xticks(range(len(chars))); ax.set_xticklabels(chars)
    ax.set_title(f"P(next | '{ctx}')", fontsize=11)
    plt.show()

def show_curves(losses, val):
    """The minibatch loss (mean of 100 steps) and the validation loss, every 100 steps."""
    avg = [sum(losses[s:s + 100]) / len(losses[s:s + 100]) for s in range(0, len(losses), 100)]
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot([s + 50 for s in range(0, len(losses), 100)], avg, label="minibatch loss (mean of 100 steps)")
    ax.plot([100 * (i + 1) for i in range(len(val))], val, color="crimson", label="validation loss")
    ax.axhline(math.log(27), color="gray", ls="--", lw=1); ax.set_xlabel("step"); ax.legend()
    plt.show()

def show_lr(curves):
    """The loss on all training examples, every 50 steps, for each learning rate."""
    fig, ax = plt.subplots(figsize=(7, 3))
    for lr, c in curves.items():
        ax.plot([50 * (i + 1) for i in range(len(c))], c, label=f"lr = {lr}")
    ax.axhline(math.log(27), color="gray", ls="--", lw=1); ax.set_xlabel("step"); ax.legend()
    plt.show()

import io, sys, traceback, html
import ipywidgets as W
from IPython.display import display

class Stepper:
    """Task by task: the task, what we need, what PyTorch gives us, then each cell's code and its run."""
    def __init__(self, tasks, cells, pictures, ns, slides=None):
        self.tasks, self.cells, self.pictures = tasks, cells, pictures
        self.slides = slides or {}   # cell -> the URL of its slide, which explains it in full
        self.order = [c for t in tasks for c in t["cells"]]
        self.ns, self.done = ns, set()   # the notebook's own namespace: its variables can be looked at after
        self.t, self.b = 0, 0
        self.head = W.HTML()
        self.beats = W.HTML()
        self.code = W.Textarea(layout=W.Layout(width="100%", height="14em"))
        self.code.add_class("stepper-code")
        self.back = W.Button(description="← Back")
        self.next = W.Button(description="Next →", button_style="primary")
        self.run = W.Button(description="Run ▶", button_style="success")
        self.pick = W.Dropdown(options=[(f"{i + 1}. {t['title']}", i) for i, t in enumerate(tasks)],
                               layout=W.Layout(width="26em"))
        self.out = W.Output(layout=W.Layout(border="1px solid #ddd", padding="6px", min_height="4em"))
        self.slide = W.HTML(layout=W.Layout(width="46%"))
        self.toggle = W.ToggleButton(value=True, description="Slide", icon="image",
                                     tooltip="show or hide the slide that explains this cell")
        self.toggle.observe(lambda ch: self.show(self.t, self.b), names="value")
        self.back.on_click(lambda _: self.go(-1))
        self.next.on_click(lambda _: self.go(+1))
        self.run.on_click(lambda _: self.execute(self.cur_cell(), self.code.value))
        self.pick.observe(lambda ch: self.show(ch["new"], 0) if ch["name"] == "value" else None, names="value")
        style = W.HTML("<style>.stepper-code textarea{font-family:ui-monospace,Menlo,Consolas,monospace;"
                       "font-size:15px;line-height:1.35;background:#f6f7f9}</style>")
        self.left = W.VBox([self.code, self.out], layout=W.Layout(width="54%"))
        self.ui = W.VBox([style, self.head, self.beats,
                          W.HBox([self.back, self.next, self.run, self.toggle, self.pick]),
                          W.HBox([self.left, self.slide], layout=W.Layout(width="100%"))])

    def beats_of(self, t):
        b = [("task", None), ("need", None), ("tool", None)]
        for c in self.tasks[t]["cells"]:
            b += [("code", c), ("run", c)]
        return b

    def cur_cell(self):
        k, c = self.beats_of(self.t)[self.b]
        return c

    def go(self, d):
        n = len(self.beats_of(self.t))
        if d > 0:
            if self.b + 1 < n:
                self.show(self.t, self.b + 1)
            elif self.t + 1 < len(self.tasks):
                self.show(self.t + 1, 0)
        else:
            if self.b > 0:
                self.show(self.t, self.b - 1)
            elif self.t > 0:
                self.show(self.t - 1, len(self.beats_of(self.t - 1)) - 1)

    def catch_up(self, cell):
        """Run, silently, every earlier cell not yet run, so that jumping ahead works."""
        for c in self.order[:self.order.index(cell)]:
            if c not in self.done:
                buf = io.StringIO(); old = sys.stdout; sys.stdout = buf
                try:
                    exec(compile(self.cells[c], f"<{c}>", "exec"), self.ns)
                finally:
                    sys.stdout = old
                self.done.add(c)

    def execute(self, cell, src):
        if cell is None:
            return
        self.out.clear_output()
        with self.out:
            try:
                self.catch_up(cell)
                exec(compile(src, f"<{cell}>", "exec"), self.ns)   # the code box ends with the picture
                self.done.add(cell)
            except Exception:
                traceback.print_exc()

    def show(self, t, b):
        self.t, self.b = t, b
        tk = self.tasks[t]
        k, c = self.beats_of(t)[b]
        if self.pick.value != t:
            self.pick.value = t
        self.head.value = (f"<div style='color:#777;font-size:13px;letter-spacing:.04em'>TASK {t + 1} OF "
                           f"{len(self.tasks)}</div><h2 style='color:#a3192f;margin:2px 0 6px'>{tk['title']}</h2>")
        items = []
        if b >= 1: items.append(("What we need", tk["need"], b == 1))
        if b >= 2: items.append(("In PyTorch", tk["tool"], b == 2))
        if k in ("code", "run"):
            items.append(("Code" if k == "code" else "Ran", html.escape(c), True))
        self.beats.value = "".join(
            f"<div style='border-left:4px solid {'#a3192f' if now else '#ddd'};padding:2px 10px;margin:6px 0;"
            f"font-size:17px;color:{'#111' if now else '#777'}'><div style='font-size:12px;letter-spacing:.05em;"
            f"color:{'#a3192f' if now else '#999'}'>{lab.upper()}</div>{txt}</div>" for lab, txt, now in items)
        show_code = k in ("code", "run")
        self.code.layout.display = "block" if show_code else "none"
        self.run.layout.display = "inline-flex" if show_code else "none"
        if show_code:
            body = self.cells[c] + ("\n" + self.pictures[c] if c in self.pictures else "")
            if k == "code" or self.code.value.split("\n")[0] != f"# {c}":
                self.code.value = f"# {c}\n" + body
                self.code.layout.height = f"{min(30, body.count(chr(10)) + 3) * 1.45}em"
        url = self.slides.get(c) if show_code and self.toggle.value else None
        self.slide.value = (f"<img src='{url}' style='width:100%;border:1px solid #ddd;border-radius:6px'>"
                            if url else "")
        self.slide.layout.display = "block" if url else "none"
        self.left.layout.width = "54%" if url else "100%"
        if k == "run":
            self.execute(c, self.code.value)
        elif k == "code":
            self.out.clear_output()
        else:
            self.out.clear_output()

    def start(self):
        display(self.ui)
        self.show(0, 0)

TASKS = [
{
"title": "Let's turn the 240 names into training examples",
"need": "Numbers, not letters: an id for each character, and pairs of (three ids → the next id).",
"tool": "<code>torch.tensor</code>: an array with a shape and a dtype (<code>int64</code> for ids).",
"cells": [
"The names",
"Characters and their ids",
"The examples of one name",
"All the examples as tensors"
]
},
{
"title": "Let's give each character a vector",
"need": "A table with one learnable row per character: 27 rows of 2 numbers.",
"tool": "<code>nn.Embedding(27, 2)</code>: the table, and <code>emb(X)</code> looks up the rows of the ids in <code>X</code>.",
"cells": [
"The embedding table",
"Looking up rows"
]
},
{
"title": "Let's compute the hidden layer",
"need": "Join the three rows into one vector of 6 numbers, then a linear layer and tanh.",
"tool": "<code>flatten(1)</code> joins; <code>nn.Linear(6, 64)</code> computes xW + b; <code>torch.tanh</code>.",
"cells": [
"Joining the three rows",
"A linear layer and tanh"
]
},
{
"title": "Let's put the model in one object",
"need": "Something that holds all the parameters and turns a context into 27 scores.",
"tool": "<code>nn.Module</code>: the layers in <code>__init__</code>, the computation in <code>forward</code>; <code>softmax</code> turns scores into probabilities.",
"cells": [
"The model as a module",
"Scores and probabilities"
]
},
{
"title": "Let's measure how wrong the model is",
"need": "−ln p of the character that really came next, averaged over a minibatch.",
"tool": "<code>F.cross_entropy(scores, targets)</code>: the softmax, −ln p and the average in one call.",
"cells": [
"The loss of one example",
"A minibatch",
"The loss of the minibatch"
]
},
{
"title": "Let's find how to change each parameter",
"need": "The gradient of the loss for every one of the 2,257 parameters.",
"tool": "<code>loss.backward()</code> fills <code>.grad</code> of every parameter. Gradients add up, so <code>zero_grad()</code> comes first.",
"cells": [
"The computation graph",
"Backpropagation",
"Gradients add up"
]
},
{
"title": "Let's take one step downhill",
"need": "w ← w − lr · grad for every parameter.",
"tool": "<code>torch.optim.SGD(lm.parameters(), lr=0.3)</code>; <code>opt.step()</code> does the update.",
"cells": [
"One step of SGD"
]
},
{
"title": "Let's train the model",
"need": "The step, repeated 1,000 times, and the loss on names the model never trains on.",
"tool": "A Python loop; <code>torch.no_grad()</code> evaluates without recording a graph.",
"cells": [
"The training loop",
"The embedding after training"
]
},
{
"title": "Let's use the trained model",
"need": "The probabilities of the next letter, and new names drawn from them.",
"tool": "<code>F.softmax</code> for the probabilities; <code>torch.multinomial</code> draws one letter.",
"cells": [
"The next letter",
"Sampling names"
]
},
{
"title": "Let's keep the trained model",
"need": "Save the parameters to a file and load them back.",
"tool": "<code>lm.state_dict()</code>, <code>torch.save</code>, <code>torch.load</code> and <code>load_state_dict</code>.",
"cells": [
"Saving and loading"
]
},
{
"title": "Let's choose how long and how fast to train",
"need": "Compare runs: a longer one, and three learning rates.",
"tool": "The same loop, run again with other settings.",
"cells": [
"Training longer",
"The learning rate"
]
}
]
CELLS = {
"The names": "import torch, math\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nwith open(\"names.txt\") as f:          # 240 first names, one per line\n    names = sorted(set(f.read().split()))\nprint(names[:8], \"...\", names[-3:])\ng = torch.Generator().manual_seed(0)\nperm = torch.randperm(len(names), generator=g).tolist()\nn_tr = int(0.85 * len(names))          # 85% to train\ntrain_words = [names[i] for i in perm[:n_tr]]\nval_words = [names[i] for i in perm[n_tr:]]\nprint(len(names), len(train_words), len(val_words))\nprint(train_words[:6])",
"Characters and their ids": "chars = [\".\"] + sorted(set(\"\".join(names)))\nstoi = {c: i for i, c in enumerate(chars)}\nV = len(chars)\nprint(V)\nprint(chars)\nprint(stoi[\"e\"], stoi[\"m\"], stoi[\"a\"])",
"The examples of one name": "def make_examples(words, block=3):\n    xs, ys = [], []\n    for w in words:\n        ctx = [0] * block                  # \"...\" at the start\n        for ch in w + \".\":\n            xs.append(ctx); ys.append(stoi[ch])\n            ctx = ctx[1:] + [stoi[ch]]\n    return torch.tensor(xs), torch.tensor(ys)\n\nX, Y = make_examples([\"emma\"])\nprint(X)\nprint(Y)",
"All the examples as tensors": "Xtr, Ytr = make_examples(train_words)\nXva, Yva = make_examples(val_words)\nprint(Xtr.shape, Xtr.dtype)\nprint(Ytr.shape, Ytr.dtype)\nprint(Xva.shape, Yva.shape)\nprint(Xtr[:4])",
"The embedding table": "torch.manual_seed(0)\nemb = nn.Embedding(V, 2)               # 27 rows of 2 numbers\nprint(emb.weight.shape)\nprint(emb.weight[:3])",
"Looking up rows": "x = Xtr[:2]                            # two contexts\ne = emb(x)                             # one row of the table per id\nprint(x)\nprint(e.shape)\nprint(e)",
"Joining the three rows": "e = e.flatten(1)                       # (2, 3, 2) -> (2, 6)\nprint(e.shape)\nprint(e)",
"A linear layer and tanh": "hidden = nn.Linear(6, 64)              # W1: (64, 6), b1: (64,)\nprint(hidden.weight.shape, hidden.bias.shape)\nh = torch.tanh(hidden(e))              # e @ W1.T + b1, then tanh\nprint(h.shape)\nprint(h[0, :5])",
"The model as a module": "class MLPLM(nn.Module):\n    def __init__(self, V=27, d=2, block=3, hidden=64):\n        super().__init__()\n        self.emb = nn.Embedding(V, d)\n        self.hidden = nn.Linear(block * d, hidden)\n        self.out = nn.Linear(hidden, V)\n    def forward(self, ctx):                         # ctx: (B, 3) ids\n        e = self.emb(ctx).flatten(1)                # (B, 6)\n        h = torch.tanh(self.hidden(e))              # (B, 64)\n        return self.out(h)                          # (B, 27) scores\n\ntorch.manual_seed(0)\nlm = MLPLM()\nfor name, p in lm.named_parameters():\n    print(name, tuple(p.shape))\nprint(sum(p.numel() for p in lm.parameters()), \"numbers\")",
"Scores and probabilities": "logits = lm(Xtr[:1])                   # the context \"...\"\np = F.softmax(logits, dim=1)\nprint(logits.shape)\nprint(logits[0, :5])\nprint(p[0, :5])\nprint(p.sum())",
"The loss of one example": "y = Ytr[:1]                            # the correct next character\nprint(chars[y.item()], p[0, y])\nprint(-torch.log(p[0, y]))\nprint(F.cross_entropy(logits, y))      # softmax, then -log",
"A minibatch": "torch.manual_seed(1)\nidx = torch.randint(0, len(Xtr), (64,))   # 64 random examples\nprint(idx[:10])\nprint(Xtr[idx].shape, Ytr[idx].shape)",
"The loss of the minibatch": "loss = F.cross_entropy(lm(Xtr[idx]), Ytr[idx])   # the mean over 64\nprint(loss)\nprint(math.log(V))                     # the loss of a uniform guess",
"The computation graph": "print(loss.grad_fn.name())            # the last operation\nprint(loss.grad_fn.next_functions[0][0].name())\nprint(lm.emb.weight.requires_grad)\nprint(lm.emb.weight.grad)              # nothing yet",
"Backpropagation": "loss.backward()                        # fills .grad of every parameter\nfor name, p in lm.named_parameters():\n    print(name, tuple(p.grad.shape), round(p.grad.norm().item(), 4))\nused = lm.emb.weight.grad.abs().sum(1) > 0\nprint([c for c, u in zip(chars, used) if not u], \"have no gradient\")",
"Gradients add up": "g1 = lm.out.bias.grad.clone()\nloss = F.cross_entropy(lm(Xtr[idx]), Ytr[idx])\nloss.backward()                        # a second time: added to .grad\nprint(torch.allclose(lm.out.bias.grad, 2 * g1))\nopt = torch.optim.SGD(lm.parameters(), lr=0.3)\nopt.zero_grad()                        # every .grad back to None\nprint(lm.out.bias.grad)",
"One step of SGD": "loss = F.cross_entropy(lm(Xtr[idx]), Ytr[idx])\nopt.zero_grad()\nloss.backward()\nw = lm.emb.weight[stoi[\"e\"]].detach().clone()\ngw = lm.emb.weight.grad[stoi[\"e\"]].clone()\nopt.step()                             # w <- w - lr * grad\nprint(w, gw)\nprint(w - 0.3 * gw)\nprint(lm.emb.weight[stoi[\"e\"]].detach())\nprint(loss.item(), F.cross_entropy(lm(Xtr[idx]), Ytr[idx]).item())",
"The training loop": "losses, val = [], []\nfor step in range(1, 1001):\n    idx = torch.randint(0, len(Xtr), (64,))\n    loss = F.cross_entropy(lm(Xtr[idx]), Ytr[idx])\n    opt.zero_grad()\n    loss.backward()\n    opt.step()\n    losses.append(loss.item())\n    if step % 100 == 0:\n        with torch.no_grad():          # no graph: we only evaluate\n            val.append(F.cross_entropy(lm(Xva), Yva).item())\n        print(step, round(loss.item(), 3), round(val[-1], 3))",
"The embedding after training": "E = lm.emb.weight.detach()\nfor c in \"aeiou.\":\n    print(c, E[stoi[c]])\nprint(E[[stoi[c] for c in \"bcdfg\"]])",
"The next letter": "ctx = torch.tensor([[stoi[c] for c in \"emm\"]])\nwith torch.no_grad():\n    p = F.softmax(lm(ctx), dim=1)\ntop = p[0].topk(5)\nfor q, i in zip(top.values, top.indices):\n    print(chars[i], round(q.item(), 3))",
"Sampling names": "g = torch.Generator().manual_seed(1)\nfor _ in range(8):\n    ctx, word = [0, 0, 0], \"\"\n    while True:\n        with torch.no_grad():\n            p = F.softmax(lm(torch.tensor([ctx])), dim=1)\n        i = torch.multinomial(p[0], 1, generator=g).item()\n        if i == 0 or len(word) > 15:\n            break\n        word += chars[i]; ctx = ctx[1:] + [i]\n    print(word, \"(a training name)\" if word in train_words else \"\")",
"Saving and loading": "torch.save(lm.state_dict(), \"mlp_lm.pt\")\nstate = torch.load(\"mlp_lm.pt\")\nprint(list(state.keys()))\nlm2 = MLPLM()\nlm2.load_state_dict(state)\nprint(torch.equal(lm2(Xva[:5]), lm(Xva[:5])))",
"Training longer": "for step in range(1001, 6001):\n    idx = torch.randint(0, len(Xtr), (64,))\n    loss = F.cross_entropy(lm(Xtr[idx]), Ytr[idx])\n    opt.zero_grad(); loss.backward(); opt.step()\n    losses.append(loss.item())\n    if step % 100 == 0:\n        with torch.no_grad():\n            val.append(F.cross_entropy(lm(Xva), Yva).item())\n    if step % 1000 == 0:\n        print(step, round(loss.item(), 3), round(val[-1], 3))\nbest = min(range(len(val)), key=val.__getitem__)\nprint(\"lowest validation loss\", round(val[best], 3), \"at step\", 100 * (best + 1))",
"The learning rate": "curves = {}\nfor lr in [0.03, 0.3, 3.0]:\n    torch.manual_seed(0)\n    m = MLPLM()\n    o = torch.optim.SGD(m.parameters(), lr=lr)\n    curves[lr] = []\n    for step in range(1, 1001):\n        idx = torch.randint(0, len(Xtr), (64,))\n        loss = F.cross_entropy(m(Xtr[idx]), Ytr[idx])\n        o.zero_grad(); loss.backward(); o.step()\n        if step % 50 == 0:\n            with torch.no_grad():\n                curves[lr].append(F.cross_entropy(m(Xtr), Ytr).item())\n    print(lr, round(curves[lr][-1], 3))"
}
PICTURES = {
"The embedding table": "show_emb(emb.weight)",
"Looking up rows": "show_emb(emb.weight, mark=x.reshape(-1))",
"The model as a module": "show_emb(lm.emb.weight)",
"Scores and probabilities": "show_bars(p[0], '...', Ytr[0])",
"The loss of one example": "show_bars(F.softmax(logits, 1)[0], '...', Ytr[0])",
"Backpropagation": "show_emb(lm.emb.weight, grad=lm.emb.weight.grad)",
"One step of SGD": "show_emb(lm.emb.weight, mark=[stoi['e']])",
"The training loop": "show_curves(losses, val)",
"The embedding after training": "show_emb(lm.emb.weight)",
"The next letter": "show_bars(p[0], 'emm')",
"Training longer": "show_curves(losses, val)",
"The learning rate": "show_lr(curves)"
}
SLIDES = {
"The names": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-04.png",
"Characters and their ids": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-05.png",
"The examples of one name": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-06.png",
"All the examples as tensors": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-07.png",
"The embedding table": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-09.png",
"Looking up rows": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-10.png",
"Joining the three rows": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-11.png",
"A linear layer and tanh": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-12.png",
"The model as a module": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-13.png",
"Scores and probabilities": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-14.png",
"The loss of one example": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-15.png",
"A minibatch": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-17.png",
"The loss of the minibatch": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-18.png",
"The computation graph": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-19.png",
"Backpropagation": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-20.png",
"Gradients add up": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-21.png",
"One step of SGD": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-22.png",
"The training loop": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-24.png",
"The embedding after training": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-25.png",
"The next letter": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-26.png",
"Sampling names": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-27.png",
"Saving and loading": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-28.png",
"Training longer": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-29.png",
"The learning rate": "https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/week09_mlp_lm_demo_files/slide-30.png"
}


In [ ]:
demo = Stepper(TASKS, CELLS, PICTURES, globals(), SLIDES)
demo.start()